In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/shubhangig870@gmail.com/sephoraa/1_Setup/Utility

In [0]:
dbutils.widgets.text("catalog","sephoraa","catalog")
dbutils.widgets.text("data_source","categories","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)

In [0]:
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# category_id
from pyspark.sql.functions import col, count, when

new=df_bronze.filter(col("category_id").rlike("^CAT"))
display(new)

In [0]:
#  category_name
# replece
df_silver = new.withColumn("category_name",when(col("category_name").cast("string")=="#N/A","unknown").otherwise(col("category_name")))
display(df_silver)

df_silver = new.withColumn("category_name",when(col("category_name").isNull(),"unknown").otherwise(col("category_name")))
display(df_silver)

df_silver = new.groupBy("category_name").count().filter(col("count")>1)
display(df_silver)

df_silver = new.filter(col("category_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_silver)


In [0]:
#  parent_category
# Null record count
from pyspark.sql.functions import col, count, when
df_silver=new.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns])
display(df_silver)

In [0]:
#  created_at
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = new.withColumn(
    "created_at",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("created_at")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("created_at")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("created_at")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("created_at")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = new.withColumn("created_at", F.to_date("created_at"))
display(df_silver)

from pyspark.sql.functions import col, when, current_date

df_silver = new.withColumn(
    "created_at",
    when(
        col("created_at").isNull(),
        current_date()
    ).otherwise(col("created_at"))
)

display(df_silver)

In [0]:
#  updated_at
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = new.withColumn(
    "updated_at",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("updated_at")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("updated_at")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("updated_at")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = new.withColumn("updated_at", F.to_date("updated_at"))
# display(df_silver)

from pyspark.sql.functions import col, when, current_date

df_silver = new.withColumn(
    "updated_at",
    when(
        col("updated_at").isNull(),
        current_date()
    ).otherwise(col("updated_at"))
)

display(df_silver)

In [0]:
#ingestion_date                                                                                                                             

In [0]:

# current_date
#  read_timestamp
#  file_name',
#  file_size